In [14]:
%matplotlib inline

import time

import numpy as np
import torch
from torch import nn
from d2l import torch as d2l

#    Statistical Efficiency
# SGD  ←──────────────→  Full GD
# noise 큼               noise 작음
# update 많음            update 적음
#
#        Computational Efficiency
# 작은 operation 반복       큰 Matrix operation
# hardware 활용 낮음        hardware 활용 높음
#
#            Minibatch SGD
#         두 효율 사이의 절충

In [15]:
# Timer

class Timer:

    def __init__(
        self,
    ) -> None:
        self.times: list[float] = []
        self.start_time = 0.0

        self.start()

    def start(
        self,
    ) -> None:
        self.start_time = time.perf_counter()

    def stop(
        self,
    ) -> float:
        elapsed_time = (
            time.perf_counter()
            - self.start_time
        )

        self.times.append(
            elapsed_time
        )

        return elapsed_time

    def average(
        self,
    ) -> float:
        return (
            sum(self.times)
            / len(self.times)
        )

    def total(
        self,
    ) -> float:
        return sum(
            self.times
        )

    def cumulative_sum(
        self,
    ) -> list[float]:
        cumulative_times = np.cumsum(
            np.asarray(
                self.times,
                dtype=np.float64,
            )
        )

        return [
            float(value)
            for value in cumulative_times
        ]

In [16]:
# Matrix 생성

matrix_size = 256

A = torch.zeros(
    matrix_size,
    matrix_size,
)

B = torch.randn(
    matrix_size,
    matrix_size,
)

C = torch.randn(
    matrix_size,
    matrix_size,
)

timer = Timer()

print("A:", tuple(A.shape))
print("B:", tuple(B.shape))
print("C:", tuple(C.shape))

A: (256, 256)
B: (256, 256)
C: (256, 256)


In [17]:
# 1. Element-wise Multiplication

timer.start()

for row in range(matrix_size):
    for column in range(matrix_size):
        A[
            row,
            column,
        ] = torch.dot(
            B[row, :],
            C[:, column],
        )
        
elementwise_time = timer.stop()
elementwise_result = A.clone()

print(
    f"Element-wise: "
    f"{elementwise_time:.6f} seconds"
)

Element-wise: 0.749888 seconds


In [18]:
# 2. Column-wise Multiplication

timer.start()

for column in range(matrix_size):
    A[:, column] = (
        B @ C[:, column]
    )
    
columnwise_time = timer.stop()
columnwise_result = A.clone()

print(
    f"Column-wise: "
    f"{columnwise_time:.6f} seconds"
)

Column-wise: 0.005643 seconds


In [19]:
# 3. Full Matrix Multiplication

timer.start()

# [256, 256]
full_result = B @ C

full_time = timer.stop()

print(
    f"Full matrix: "
    f"{full_time:.6f} seconds"
)

Full matrix: 0.001142 seconds


In [20]:
# Performance 비교

# 작은 Floating-Point 오차를 허용하고 세 결과가 같은지 검증
torch.testing.assert_close(
    elementwise_result,
    full_result,
    rtol=1e-3,
    atol=1e-4,
)

torch.testing.assert_close(
    columnwise_result,
    full_result,
    rtol=1e-3,
    atol=1e-4,
)

# m×n Matrix와 n×p Matrix Multiplication에는
# 대략 2mnp번의 Floating-Point Operation이 필요
num_operations = (
    2 * matrix_size**3
)

gigaflops = {
    "Element-wise": (
        num_operations
        / elementwise_time
        / 1e9
    ),
    "Column-wise": (
        num_operations
        / columnwise_time
        / 1e9
    ),
    "Full matrix": (
        num_operations
        / full_time
        / 1e9
    ),
}

for method, performance in gigaflops.items():
    print(
        f"{method:<14}: "
        f"{performance:>8.3f} GFLOPS"
    )

Element-wise  :    0.045 GFLOPS
Column-wise   :    5.946 GFLOPS
Full matrix   :   29.369 GFLOPS


In [ ]:
# 4. Block Matrix Multiplication

block_size = 64

A = torch.zeros_like(
    full_result
)

timer.start()

# (0, 64, 128, 192)
for start_column in range(
    0,
    matrix_size, # 256
    block_size,  # 64
):
    # (64, 128, 192, 256)
    end_column = min(
        start_column + block_size,
        matrix_size,        
    )
    
    A[
        :,
        start_column:end_column,
    ] = (
        B
        @ C[
            :,
            start_column:end_column,
        ]
    )

block_time = timer.stop()

torch.testing.assert_close(
    A,
    full_result,
    rtol=1e-3,
    atol=1e-4,
)

block_gigaflops = (
    num_operations
    / block_time
    / 1e9
)

print(
    f"Block matrix: "
    f"{block_time:.6f} seconds"
)

print(
    f"Block matrix: "
    f"{block_gigaflops:.3f} GFLOPS"
)

Block matrix: 0.004369 seconds
Block matrix: 7.680 GFLOPS
